# RetailFlow - Gold Layer
Transforms cleaned Silver datasets into business-ready analytical data models and KPIs for Power BI.

In [ ]:
from pathlib import Path
import pandas as pd
import re

# Dynamically resolve project root from any folder or subfolder
current = Path.cwd()
project_root = None
for candidate in [current, current / 'RetailFlow', *current.parents]:
    if (candidate / 'data' / 'raw').exists():
        project_root = candidate
        break
    if (candidate / 'RetailFlow' / 'data' / 'raw').exists():
        project_root = candidate / 'RetailFlow'
        break
if project_root is None:
    raise FileNotFoundError('RetailFlow root directory with data/raw not found.')

SILVER_PATH = project_root / 'data' / 'silver'
GOLD_PATH = project_root / 'data' / 'gold'
PBI_PATH = project_root / 'powerbi' / 'data'

GOLD_PATH.mkdir(parents=True, exist_ok=True)
PBI_PATH.mkdir(parents=True, exist_ok=True)

# Load all Silver datasets
data = {}
for file in SILVER_PATH.glob('*.parquet'):
    name = file.stem.replace('_dataset', '')
    data[name] = pd.read_parquet(file)

print(f'Silver datasets loaded: {len(data)}')
for name, df in sorted(data.items()):
    print(f'  {name:<35} : {df.shape}')


In [ ]:
orders = data['olist_orders'].copy()
customers = data['olist_customers'].copy()
payments = data['olist_order_payments'].copy()
items = data.get('olist_order_items')
products = data.get('olist_products')
categories = data.get('product_category_name_translation')

# 1. Order-level item aggregation
if items is not None:
    item_summary = items.groupby('order_id').agg(
        total_items=('order_item_id', 'count'),
        product_revenue=('price', 'sum'),
        shipping_cost=('freight_value', 'sum')
    ).reset_index()
else:
    item_summary = pd.DataFrame(columns=['order_id', 'total_items', 'product_revenue', 'shipping_cost'])

# 2. Order-level payment aggregation
payment_summary = payments.groupby('order_id').agg(
    payment_total=('payment_value', 'sum'),
    primary_payment_type=('payment_type', 'first'),
    payment_installments=('payment_installments', 'max')
).reset_index()

# Merge order details
gold_orders = orders.merge(item_summary, on='order_id', how='left')
gold_orders = gold_orders.merge(payment_summary, on='order_id', how='left')
gold_orders = gold_orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
    on='customer_id', how='left'
)

# Fill defaults and fallback revenue
gold_orders['payment_total'] = gold_orders['payment_total'].fillna(0.0)
if items is not None:
    gold_orders['total_items'] = gold_orders['total_items'].fillna(1)
    gold_orders['product_revenue'] = gold_orders['product_revenue'].fillna(gold_orders['payment_total'] * 0.85)
    gold_orders['shipping_cost'] = gold_orders['shipping_cost'].fillna(gold_orders['payment_total'] * 0.15)
else:
    gold_orders['total_items'] = 1
    gold_orders['product_revenue'] = gold_orders['payment_total'] * 0.85
    gold_orders['shipping_cost'] = gold_orders['payment_total'] * 0.15

# Convert datetime columns
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    gold_orders[col] = pd.to_datetime(gold_orders[col], errors='coerce')

gold_orders['delivery_days'] = (
    gold_orders['order_delivered_customer_date'] - gold_orders['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)
gold_orders['is_delayed'] = (
    gold_orders['order_delivered_customer_date'] > gold_orders['order_estimated_delivery_date']
).astype('Int64')

print('Business data prepared!')
print(f'Total orders: {len(gold_orders):,}')


In [ ]:
# Delivered orders for sales and revenue analysis
delivered = gold_orders[gold_orders['order_status'] == 'delivered'].copy()
delivered['sales_month'] = delivered['order_purchase_timestamp'].dt.to_period('M').astype(str)

# 1. Order Summary Table
gold_order_summary = gold_orders.copy()

# 2. Monthly Sales Table
gold_monthly_sales = delivered.groupby('sales_month').agg(
    total_orders=('order_id', 'nunique'),
    total_customers=('customer_unique_id', 'nunique'),
    total_revenue=('product_revenue', 'sum'),
    total_shipping=('shipping_cost', 'sum'),
    total_payment=('payment_total', 'sum'),
    avg_order_value=('payment_total', 'mean')
).reset_index().round(2)

# 3. Category Sales Table
if items is not None and products is not None:
    cat_items = items.merge(orders[['order_id', 'order_status']], on='order_id', how='inner')
    cat_items = cat_items[cat_items['order_status'] == 'delivered']
    cat_items = cat_items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    if categories is not None:
        cat_items = cat_items.merge(categories, on='product_category_name', how='left')
        cat_items['category'] = cat_items['product_category_name_english'].fillna(cat_items['product_category_name']).fillna('other')
    else:
        cat_items['category'] = cat_items['product_category_name'].fillna('other')

    gold_category_sales = cat_items.groupby('category').agg(
        total_items_sold=('order_item_id', 'count'),
        total_orders=('order_id', 'nunique'),
        total_revenue=('price', 'sum'),
        avg_item_price=('price', 'mean')
    ).reset_index().rename(columns={'category': 'product_category_name_english'}).round(2)
    gold_category_sales = gold_category_sales.sort_values(by='total_revenue', ascending=False).reset_index(drop=True)
else:
    gold_category_sales = pd.DataFrame(columns=['product_category_name_english', 'total_items_sold', 'total_orders', 'total_revenue', 'avg_item_price'])

# 4. State Sales Table
gold_state_sales = delivered.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    total_customers=('customer_unique_id', 'nunique'),
    total_revenue=('product_revenue', 'sum'),
    total_shipping=('shipping_cost', 'sum'),
    total_payment=('payment_total', 'sum'),
    avg_order_value=('payment_total', 'mean')
).reset_index().round(2).sort_values(by='total_revenue', ascending=False).reset_index(drop=True)

# 5. Payment Analysis Table
pay_orders = payments.merge(orders[['order_id', 'order_status']], on='order_id', how='inner')
pay_delivered = pay_orders[pay_orders['order_status'] == 'delivered']
gold_payment_analysis = pay_delivered.groupby('payment_type').agg(
    total_transactions=('order_id', 'count'),
    total_orders=('order_id', 'nunique'),
    total_payment_value=('payment_value', 'sum'),
    avg_transaction_value=('payment_value', 'mean')
).reset_index().round(2).sort_values(by='total_payment_value', ascending=False).reset_index(drop=True)

print('All 5 Gold analytical tables created successfully!')


In [ ]:
gold_tables = {
    'gold_order_summary': gold_order_summary,
    'gold_monthly_sales': gold_monthly_sales,
    'gold_category_sales': gold_category_sales,
    'gold_state_sales': gold_state_sales,
    'gold_payment_analysis': gold_payment_analysis
}

for name, df in gold_tables.items():
    # Save Parquet for Lakehouse
    df.to_parquet(GOLD_PATH / f'{name}.parquet', index=False)
    # Save CSV for Power BI
    df.to_csv(GOLD_PATH / f'{name}.csv', index=False)
    df.to_csv(PBI_PATH / f'{name}.csv', index=False)
    print(f'Saved: {name:<25} | Rows: {len(df):>7,} | Parquet & CSV')

print('\nGOLD LAYER COMPLETED!')


In [ ]:
for name, df in gold_tables.items():
    print(f'\n--- {name} ---')
    try:
        display(df.head(3))
    except Exception:
        print(df.head(3).to_string(index=False))
